In [ ]:
# 本文件在monthly_4的基础上通过对signal变量的分组进一步消除stock_code维度，得到F矩阵（T*K）
# 采用两种分组方法：unconditional sort（对应单变量分组）和size_neutralized sort（对应市值加权分组）
%run config_0.ipynb
%run data_io_1.ipynb
%run clean_and_universe_2.ipynb
%run features_3.ipynb
%run monthly_4.ipynb

import pandas as pd
import numpy as np

In [ ]:
# 在月度面板表中进行单变量分组，单变量分组为10组

# 规定做多做空方向，假设ascending = False
monthly_signal = [
        monthly_market_value ,
        monthly_bm ,
        monthly_mom ,
        monthly_cp , 
        monthly_asset_growth ,
        monthly_volatility ,
        monthly_max ,
        monthly_illiquidity ,
        monthly_turnover_12m ,
        monthly_abn_turnover_1m
    ]

direction_dict = {
    monthly_market_value : {
        'long_leg' : 1 , 
        'short_leg' : 10 , 
        'group_name' : 'mv_G' , 
        'factor_name' : smb
    } , 
    monthly_bm : {
        'long_leg' : 10 , 
        'short_leg' : 1 , 
        'group_name' : 'bm_G' , 
        'factor_name' : hml
    } , 
    monthly_mom : {
        'long_leg' : 10 , 
        'short_leg' : 1 , 
        'group_name' : 'mom_G' , 
        'factor_name' : mom_fac
    } , 
    monthly_cp : {
        'long_leg' : 10 , 
        'short_leg' : 1 ,
        'group_name' : 'cp_G' , 
        'factor_name' : cp_fac
    } ,
    monthly_asset_growth : {
        'long_leg' : 1 , 
        'short_leg' : 10 , 
        'group_name' : 'ag_G' , 
        'factor_name' : cma
    } , 
    monthly_volatility : {
        'long_leg' : 1 , 
        'short_leg' : 10 , 
        'group_name' : 'vol_G' , 
        'factor_name' : vol_fac
    } , 
    monthly_max : {
        'long_leg' : 1 , 
        'short_leg' : 10 , 
        'group_name' : 'max_G' , 
        'factor_name' : max_fac
    } , 
    monthly_illiquidity : {
        'long_leg' : 10 , 
        'short_leg' : 1 , 
        'group_name' : 'illiq_G' , 
        'factor_name' : illiq_fac
    } , 
    monthly_turnover_12m : {
        'long_leg' : 1 , 
        'short_leg' : 10 , 
        'group_name' : 'turn_G' , 
        'factor_name' : turn_fac
    } , 
    monthly_abn_turnover_1m : {
        'long_leg' : 1 , 
        'short_leg' : 10 , 
        'group_name' : 'abturn_G' , 
        'factor_name' : abn_turn_fac
    }
}

In [ ]:
# 对风格因子进行分组、排序操作
def get_groups_uncon(monthly_final):
    for signal , info in direction_dict.items():
        group_name = info['group_name']
        long_leg = info['long_leg']
        short_leg = info['short_leg']
        factor_name = info['factor_name']
        monthly_final[group_name] = monthly_final.groupby(
            ym
        )[signal].transform(lambda x:pd.qcut(x , q = 10 , labels = False , duplicates = 'drop') + 1)

        long_ret = monthly_final[monthly_final[group_name] == long_leg].groupby(ym)[monthly_excess_ret].mean()
        short_ret = monthly_final[monthly_final[group_name] == short_leg].groupby(ym)[monthly_excess_ret].mean()
        monthly_final[factor_name] = monthly_final[ym].map(long_ret - short_ret)
        print('查看{}的long_ret'.format(signal))
        print(long_ret.head())
        print(long_ret.shape)
        print('查看{}的short_ret'.format(signal))
        print(short_ret.head())
        print(short_ret.shape)
        print('查看{}的factor_name'.format(signal))
        print(monthly_final[factor_name].head())
        print(monthly_final.shape)
        
    # 市场因子：每个月的（等权）平均超额收益
    monthly_final[mkt] = monthly_final.groupby(ym)[monthly_excess_ret].transform('mean')
    # F矩阵：T(ym) × K(因子)，每月一行、每因子一列
    F = monthly_final.groupby(ym)[[mkt , smb , hml , mom_fac , cma , vol_fac , max_fac , illiq_fac , turn_fac , abn_turn_fac]].first()
    return F

df = read_daily_data()
df = cleaning(df)
df = universe(df)
cf_df = read_cf_data()
cf_df = cf_cleaning(cf_df)
bs_df = read_bs_data()
bs_df = bs_cleaning(bs_df)
df = get_market_feature(df)
df = get_momentum(df)
df = get_cp(df , cf_df)
df = get_asset_growth(df , bs_df)
df = get_excess_return(df)
df = get_bm(df)
df = get_ep(df)
df = get_one_month_volatility(df)
df = get_max_return(df)
df = get_illiquidity(df)
df = get_turnover_12m(df)
df = get_abn_turnover_1m(df)
monthly = get_ym_monthly_excessret(df)
snap = get_form_signal(df)
monthly_final = get_monthly_panel(monthly , snap)
F = get_groups_uncon(monthly_final)
print('查看单变量分组结果')
print(F.head())
print(F.shape)

In [ ]:
# 在月度面板表中进行双变量分组，另一个取市值，每个划分5组，共25组
def get_groups_neutral(monthly_final):
    n = 5  # 每个维度分5组
    size_group = 'size_G'  # 临时列：每月市值五分位组号

    for signal , info in direction_dict.items():
        group_name = info['group_name']
        long_leg = info['long_leg']
        short_leg = info['short_leg']
        factor_name = info['factor_name']

        # direction_dict 里的 1/10 表示最低/最高组，5分组下映射到 1/5
        long_g = n if long_leg == 10 else 1
        short_g = n if short_leg == 10 else 1

        # 1) 每月按市值分5组（rank 让值唯一；duplicates='drop' 兼容全 NaN 的月份）
        monthly_final[size_group] = monthly_final.groupby(ym)[monthly_market_value].transform(
            lambda x: pd.qcut(x.rank(method = 'first') , q = n , labels = False , duplicates = 'drop') + 1
        )

        # 2) 每月、每个市值组内，再按因子分5组
        monthly_final[group_name] = monthly_final.groupby([ym , size_group])[signal].transform(
            lambda x: pd.qcut(x.rank(method = 'first') , q = n , labels = False , duplicates = 'drop') + 1
        )

        # 3) 组内多空收益：每月、每市值组
        long_ret = monthly_final[monthly_final[group_name] == long_g].groupby([ym , size_group])[monthly_excess_ret].mean()
        short_ret = monthly_final[monthly_final[group_name] == short_g].groupby([ym , size_group])[monthly_excess_ret].mean()

        # 4) 对市值组取平均 → 市值中性因子收益（每月一个值）
        ls = (long_ret - short_ret).groupby(ym).mean()
        monthly_final[factor_name] = monthly_final[ym].map(ls)

        print('查看{}的long_ret'.format(signal))
        print(long_ret.head())
        print('查看{}的short_ret'.format(signal))
        print(short_ret.head())
        print('查看{}的ls'.format(signal))
        print(ls)
        print('查看{}的factor_name'.format(signal))
        print(monthly_final[factor_name].head())

    # 市场因子 + F矩阵（与单变量分组一致）
    monthly_final[mkt] = monthly_final.groupby(ym)[monthly_excess_ret].transform('mean')
    F = monthly_final.groupby(ym)[[mkt , smb , hml , mom_fac , cma , vol_fac , max_fac , illiq_fac , turn_fac , abn_turn_fac]].first()
    return F

df = read_daily_data()
df = cleaning(df)
df = universe(df)
cf_df = read_cf_data()
cf_df = cf_cleaning(cf_df)
bs_df = read_bs_data()
bs_df = bs_cleaning(bs_df)
df = get_market_feature(df)
df = get_momentum(df)
df = get_cp(df , cf_df)
df = get_asset_growth(df , bs_df)
df = get_excess_return(df)
df = get_bm(df)
df = get_ep(df)
df = get_one_month_volatility(df)
df = get_max_return(df)
df = get_illiquidity(df)
df = get_turnover_12m(df)
df = get_abn_turnover_1m(df)
monthly = get_ym_monthly_excessret(df)
snap = get_form_signal(df)
monthly_final = get_monthly_panel(monthly , snap)
F = get_groups_neutral(monthly_final)
print('查看双变量市值加权分组结果')
print(F.head())
print(F.shape)

In [ ]:
# CH-3 中国三因子模型（Liu, Stambaugh & Yuan 2019）
# 剔壳30% → 市值中位数分S/B → EP 30/70分V/M/G → 2×3六组合 → 市值加权 → SMB/VMG/MKT
def get_groups_ch3(monthly_final):
    df = monthly_final.copy()

    # 1) 剔壳污染：每月剔除市值最小 30% 的股票
    df['_mv_pct'] = df.groupby(ym)[monthly_market_value].rank(pct = True)
    df = df[df['_mv_pct'] > 0.30]

    # 2) 市值中位数分 S/B
    df['_sz'] = df.groupby(ym)[monthly_market_value].transform(
        lambda x: pd.qcut(x.rank(method = 'first') , q = [0 , 0.5 , 1.0] , labels = ['S', 'B'])).astype(str)
   
    # 3) EP 的 30%/70% 分位数分 V/M/G（高 EP = 价值 V）
    df['_ep_g'] = df.groupby(ym)[monthly_ep].transform(
        lambda x: pd.qcut(x.rank(method = 'first'), q = [0, 0.3, 0.7, 1.0], labels = ['G', 'M', 'V'])).astype(str)

    # 4) 2×3 = 6 组合
    df['_port'] = df['_sz'] + '/' + df['_ep_g']

    # 5) 每个组合市值加权收益
    def vw_ret(g):
        w = g[monthly_market_value] / g[monthly_market_value].sum()
        return (g[monthly_excess_ret] * w).sum()

    port_ret = df.groupby([ym, '_port'])[[monthly_market_value, monthly_excess_ret]].apply(vw_ret).unstack()
    port_ret = port_ret.reindex(columns = ['S/V', 'S/M', 'S/G', 'B/V', 'B/M', 'B/G'])

    # 6) 合成 SMB / VMG / MKT
    SMB = (port_ret['S/V'] + port_ret['S/M'] + port_ret['S/G']) / 3 \
        - (port_ret['B/V'] + port_ret['B/M'] + port_ret['B/G']) / 3
    VMG = (port_ret['S/V'] + port_ret['B/V']) / 2 \
        - (port_ret['S/G'] + port_ret['B/G']) / 2
    MKT = df.groupby(ym)[[monthly_market_value, monthly_excess_ret]].apply(vw_ret)   # 剩余 70% 股票池的市值加权收益

    F = pd.DataFrame({mkt: MKT, smb: SMB, vmg: VMG})
    return F

# 复用上面已算好的 monthly_final（含 monthly_ep）
F_ch3 = get_groups_ch3(monthly_final)
print('查看 CH-3 三因子结果')
print(F_ch3.head())
print(F_ch3.shape)